# 07. Reproducing Repeated Contention Scheduling (RCS)

A. Rolich, M. Tricco, S. Paroli, M. Yildiz, A. Baiocchi, *Repeated Contention Scheduling: A Novel
Resource Allocation Algorithm Toward 6G Vehicular Networks*, MAIN 2026, arXiv:2607.06103.

This notebook compares MoReV2X's RCS (`docs/rcs.md`) with the paper's Section III, Figs 2 to 8,
and then runs the same schemes on MoReV2X's highway with the PHY and with tones that only nearby
UEs hear.

**Data.** Seed 1 of `tools/rcs/grid.py` on ease-rtx: 315 runs, every one finished, none failed.
`results/rcs/summary_seed1.csv` (not in git) holds one row per run. A second seed of the Figs 7 and 8 runs was
still running when this notebook was written; it is not in these numbers.

## Setup

| | Paper | Here |
|---|---|---|
| Channel | 20 MHz, SCs of 25 RBs, MCS 13 | same |
| Frame | Tf = 10 ms, 1 ms of contention | same: 1 contention slot at mu = 0, 2 at mu = 1 |
| SCs per frame K | 36 (mu = 0: 9 slots x 4, mu = 1: 18 x 2) | same |
| Sub-carriers M | 400 (mu = 0), 200 (mu = 1) | same, one out of three of the band |
| Rounds r, groups J | 3; J in {1, 2, 3, 6, 9, 18, 36} | same |
| Traffic | 200 B every 100 ms, TDB 60 ms | same (`--UniformPeriod=100 --PDB=60`) |
| Load a | n Tf / (K Tg): n = 360 a for RCS, 400 a for SPS and DS | same, a = 0.3 to 1.1 |
| Channel model | all nodes hear each other, a packet fails only by collision | `--Ideal`: static UEs in a 10 m square, no shadowing; `tools/rcs/metrics.py --model ideal` counts a packet as received when no other transmission used its slot and sub-channel |
| SPS | Ppers 0 and 0.8, RC 5 to 15 | MoReV2X's 3GPP SB-SPS with `--pkeep=0` / `0.8`, RC 5 to 15 |
| DS | random selection | MoReV2X's DS: the same sensing, one shot per packet |
| Simulated time | not stated | 50 s for the runs of Figs 7 and 8 and the highway, 20 s for the rest; the first second is not counted |

**Where the comparison is not like for like.** MoReV2X's SPS always finds a resource, so its
Pdrop is 0: the paper's SPS drops a packet when too few candidates are left. MoReV2X's DS senses,
the paper's picks at random; in this all-hear square a one-shot DS leaves no reservation to sense,
so the two behave alike.

In [ ]:
import importlib.util, os
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 200)
ROOT = os.path.abspath("../..")
spec = importlib.util.spec_from_file_location("rcs_plots", os.path.join(ROOT, "tools", "rcs", "plots.py"))
plots = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plots)

runs = pd.read_csv(os.path.join(ROOT, "results", "rcs", "summary_seed1.csv"))
FIGURES = os.path.join(ROOT, "docs", "figures", "rcs")
os.makedirs(FIGURES, exist_ok=True)
plots.j_sweep(runs, FIGURES)
plots.comparison(runs, FIGURES)
plots.comparison(runs, FIGURES, model="highway", suffix="_300m")
plots.highway_tones(runs, FIGURES)
show = lambda name: display(Image(os.path.join(FIGURES, name), width=900))

ideal = runs[runs.model == "ideal"]
rcs = ideal[ideal.scheme == "rcs"]
runs.groupby(["model", "scheme", "mu"]).agg(runs=("name", "size"), loads=("load", "nunique"), sim_s=("sim_time", "max"))

## Figs 2 to 6: RCS for each number of groups J

PRR, collision and drop probabilities, mean peak interval E[PIR] and mean age E[AoI], against
the load, for mu = 0 (left) and mu = 1 (right).

In [ ]:
for name in ["fig2_prr.png", "fig3_pcoll.png", "fig4_pdrop.png", "fig5_pir_ms.png", "fig6_aoi_ms.png"]:
    show(name)

In [ ]:
prr = rcs.pivot_table(index="load", columns=["mu", "groups"], values="prr")
print("PRR, best J at each load:")
print(pd.DataFrame({mu: prr[mu].idxmax(axis=1).astype(int) for mu in (0, 1)}).T.to_string())
print()
print("a = 1.1:")
print(rcs[rcs.load == 1.1].pivot_table(index="groups", columns="mu", values=["pcoll", "pdrop", "pir_ms"]).round(4).to_string())

### Against the paper's Section III-C

| Paper | Here | |
|---|---|---|
| At low load J = 36 has the highest PRR and the lowest Pcoll | J = 36 has the highest PRR at every load up to 0.9, for both numerologies; its Pcoll stays near 0.001 to 0.02 | matches |
| mu = 0: from a = 0.9, J = 1 becomes the best | the curves meet at a = 1.0 (all J within 0.913 +- 0.004); J = 36 is best again at 1.1 because J = 1 starts dropping | crossover at 1.0, not 0.9 |
| mu = 1: J = 36 is best at every load, J = 1 the worst | J = 36 best at every load; J = 1 to 9 within 0.03 of each other, below J = 18 up to a = 1.0 | matches |
| Pcoll lower at mu = 0 than at mu = 1 | 0.08 against 0.17 for J = 1 at a = 1.1: M = 400 against 200 sub-carriers, so fewer equal tones | matches |
| Pdrop much higher for J = 36; near 0 for J = 1 until a > 0.9; J = 36 in [0.01, 0.1] at a = 1 | J = 36: 0.089 (mu = 0) and 0.084 (mu = 1) at a = 1; J = 1: 0 up to 0.9, 0.08 (mu = 0) and 0.03 (mu = 1) at 1.1 | matches |
| Pdrop barely depends on the numerology | 0.144 against 0.137 for J = 36 at a = 1.1 | matches |
| E[PIR] spread between J of about 15 ms under high load | 3.5 ms (mu = 0) and 9 ms (mu = 1) at a = 1.1 | smaller spread |

## Fig 7: RCS against SPS and DS (mu = 1)

In [ ]:
show("fig7_comparison.png")
fig7 = ideal[ideal.mu == 1].assign(scheme=lambda f: f.scheme.where(f.scheme != "rcs", "rcs J" + f.groups.fillna(0).astype(int).astype(str)))
fig7 = fig7[fig7.scheme.isin(["rcs J1", "rcs J36", "sps0", "sps08", "ds"])]
for metric in ["prr", "pir_ms", "aoi_ms"]:
    print(metric)
    print(fig7.pivot_table(index="scheme", columns="load", values=metric).round(3).to_string())
    print()

| Paper | Here | |
|---|---|---|
| RCS outperforms SPS with Ppers = 0 and DS at every load | J = 36 does at every load. J = 1 beats DS everywhere but is below SPS Ppers = 0 up to a = 0.6 (0.946 against 0.963 at 0.3) and above it from 0.7 | J = 36 matches; J = 1 only from a = 0.7 |
| RCS matches SPS only with Ppers = 0.8, at low load | SPS 0.8 is within 0.011 of J = 36 up to a = 0.7, then falls to 0.56 at 1.1 against 0.84 | matches |
| E[PIR] grows steeply for SPS and DS around a = 0.8 to 0.9; RCS stays close to 100 ms | RCS 100 to 125 ms; SPS up to 170 ms, DS up to 300 ms | matches |
| E[AoI]: RCS far below every standard scheme; SPS worst at high load from persistent collisions | RCS 50 to 76 ms; SPS Ppers = 0.8 reaches 1.3 s at a = 1.1, DS 250 ms | matches |

## Fig 8: AoI violation probability (mu = 1)

In [ ]:
show("fig8_aoi_violation.png")
viol = fig7[fig7.load.isin([0.3, 0.7, 1.1])].pivot_table(index="scheme", columns="load",
        values=["aoi_violation_100", "aoi_violation_200", "aoi_violation_500"])
viol.round(4)

| Paper | Here | |
|---|---|---|
| RCS has the lowest violation probability at every threshold | at 200 and 500 ms, yes. At 100 ms SPS Ppers = 0.8 is lower up to a = 0.8 (0.006 against 0.03 for J = 36 at 0.3) | not at 100 ms below a = 0.9 |
| For SPS, V(Ath) changes little between thresholds | SPS 0.8 at a = 1.1: 0.42, 0.40, 0.37 for 100, 200, 500 ms | matches |
| RCS V(200 ms) below 0.1 even at high load; V(500 ms) near 0 | J = 1: 0.049, J = 36: 0.034 at a = 1.1; V(500) below 0.001 | matches |

At 100 ms the threshold equals the period, so delay jitter alone violates it. An RCS packet that
loses a contention arrives a frame or more later (J = 36 plays 1.19 contentions per packet at
a = 0.3), while SPS Ppers = 0.8 keeps sending every 100 ms on the same resource.

## What the contention does

From `RcsLog.txt` and `RcsContentionLog.txt`, mu = 1. With J = 1 nearly every packet wins its
first contention until the load passes 0.9; the losses are collisions of winners that picked the
same tones. With J = 36 a packet loses more often (one SC per group) and waits for later frames,
which costs access delay and, once the 60 ms run out, drops.

In [ ]:
cols = ["rcs_contentions_per_packet", "rcs_access_delay_ms", "rcs_contenders_per_frame",
        "rcs_winners_per_frame", "rcs_scs_collided_per_frame"]
inner = rcs[(rcs.mu == 1) & rcs.groups.isin([1, 36])]
inner.pivot_table(index="load", columns="groups", values=cols).round(2)

## On MoReV2X's highway

The same traffic on the 5 km highway, with the PHY deciding what is received (SINR, half
duplex, path loss and shadowing). The metrics count the pairs within 300 m. RCS runs once with
each way of hearing the tones (`--RcsTone`):

- `all`: every UE hears every tone, wherever it is, as the paper assumes;
- `range150`, `range300`, `range600`: tones heard within that many metres;
- `power110`, `power95`: tones heard when 23 dBm minus the path loss and shadowing reaches -110 or
  -95 dBm.

In [ ]:
show("highway_tones_J1.png")
show("highway_tones_J36.png")
show("fig7_comparison_highway_300m.png")
hw = runs[runs.model == "highway"].copy()
hw["setting"] = hw.apply(lambda r: "rcs J%d %s" % (r.groups, r.tone) if r.scheme == "rcs" else r.scheme, axis=1)
hw.pivot_table(index="setting", columns="load", values="prr_300m").round(3)

**Who hears the tones decides the result.**

- With tones heard by all, the highway behaves like the paper's square: RCS J = 36 is as good as
  SPS (0.995 against 0.992 at a = 0.3) up to a = 0.7, and then falls (0.85 at 1.1), because the
  whole 5 km contends for the same 36 SCs.
- With tones heard only nearby, RCS J = 1 breaks down: 0.65 to 0.50 within 300 m at 150 m of tone
  range. With J = 1 a winner takes the SC numbered by the lower tones it heard, so two UEs that do
  not hear each other often both take SC 0 or 1, and a receiver between them loses both.
- With J = 36 the random group spreads those UEs over the frame, so reuse between UEs that cannot
  hear each other mostly works. With tones detected at -95 dBm, RCS J = 36 holds 0.984 at a = 1.1,
  the best of all the schemes on the highway (SPS Ppers = 0.8: 0.982, DS: 0.900).
- On the highway the sensing schemes work much better than in the square, since most interferers
  are far: SPS stays above 0.97 and DS above 0.90 at every load.

## Summary

- The ideal model reproduces the paper's trends: J = 36 is best at mu = 1 and at low load, Pdrop
  grows with J, Pcoll with the load and with fewer sub-carriers, and RCS keeps E[AoI] near 50 to
  75 ms where SPS reaches seconds.
- Three differences: the mu = 0 crossover between J = 1 and J = 36 is at a = 1.0 instead of 0.9;
  RCS J = 1 is below SPS Ppers = 0 at low load; and at Ath = 100 ms SPS Ppers = 0.8 has the lower
  violation probability below a = 0.9. The paper's SPS and DS are not MoReV2X's (see Setup), and
  this is one seed.
- The paper's results need every UE to hear every tone. When tones are heard only nearby, J = 1
  suffers hidden-terminal collisions, and J = 36 with a power-based detection does best.

To regenerate: `tools/rcs/grid.py --summary`, keep the seed 1 rows of `results/rcs/summary.csv`
as `results/rcs/summary_seed1.csv` (not in git) and run this notebook.